# Computer Vision Fundamentals with PyTorch

In this notebook, you will:

1. Understand how a digital image is represented.
2. Load and visualize CIFAR-10 using a `Dataset` and `DataLoader`.
3. build a ResNet-18 classifier for 10 classes.
4. understand the loss function, optimizer, and training loop.
5. train, validate, and test the model.
6. run a pretrained YOLO detector to see how object detection differs from classification.

> **Recommended runtime:** Google Colab with GPU  
> In Colab: **Runtime → Change runtime type → T4 GPU**

## 0. Setup

PyTorch is usually preinstalled in Google Colab. The first cell checks the environment and makes the experiment reproducible.

A **random seed** helps us obtain similar results when the notebook is run again.

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.transforms import functional as TF

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Part 1 — What is an image?

A digital image is a grid of pixels.

For a color image, each pixel usually contains three values:

- **R**: red
- **G**: green
- **B**: blue

For a grayscale image, each pixel contains a single intensity value.

In PyTorch, an image is commonly stored as a tensor with shape:

```text
channels × height × width
```

In this section, we will use **higher-resolution sample images**:

- a grayscale image
- an RGB image



## 1.1 Load one grayscale image and one RGB image

We use:

- `camera` for a grayscale image
- `astronaut` for an RGB image

Both are higher resolution than CIFAR-10 and are better for introducing image structure and visualization.


In [ ]:
from skimage import data

gray_np = data.camera()        # shape: H x W
rgb_np = data.astronaut()      # shape: H x W x 3

gray_tensor = torch.from_numpy(gray_np).float() / 255.0
gray_tensor = gray_tensor.unsqueeze(0)  # 1 x H x W

rgb_tensor = torch.from_numpy(rgb_np).float() / 255.0
rgb_tensor = rgb_tensor.permute(2, 0, 1)  # 3 x H x W

print("Grayscale tensor shape:", gray_tensor.shape)
print("RGB tensor shape:", rgb_tensor.shape)
print("Grayscale min/max:", gray_tensor.min().item(), gray_tensor.max().item())
print("RGB min/max:", rgb_tensor.min().item(), rgb_tensor.max().item())

## 1.2 Visualize the grayscale and RGB images

Notice the difference:

- the grayscale image has **1 channel**
- the RGB image has **3 channels**


In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(gray_tensor.squeeze(0), cmap="gray")
plt.title("Grayscale image")
plt.axis("off")
plt.show()

plt.figure(figsize=(6, 6))
plt.imshow(rgb_tensor.permute(1, 2, 0))
plt.title("RGB image")
plt.axis("off")
plt.show()

## 1.3 Inspect pixel values

The model does not initially see “camera” or “astronaut.”  
It receives numbers.

We now inspect small patches from each image.


In [ ]:
gray_patch = gray_tensor[0, :5, :5]
rgb_patch = rgb_tensor[:, :3, :3]

print("Grayscale 5x5 patch (scaled to 0-255):")
print(torch.round(gray_patch * 255).to(torch.int32))

print("\nRGB 3x3 patch (scaled to 0-255):")
for channel_name, channel_values in zip(["Red", "Green", "Blue"], rgb_patch):
    print(f"\n{channel_name} channel:")
    print(torch.round(channel_values * 255).to(torch.int32))

## 1.4 Visualize the RGB channels

Instead of displaying each channel as a gray image, we keep the **color identity** of each channel.

- Red channel shown in red
- Green channel shown in green
- Blue channel shown in blue

This makes it easier to understand how the three channels combine to form the final RGB image.


In [ ]:
def isolate_rgb_channel(image_tensor, channel_index):
    isolated = torch.zeros_like(image_tensor)
    isolated[channel_index] = image_tensor[channel_index]
    return isolated

red_only = isolate_rgb_channel(rgb_tensor, 0)
green_only = isolate_rgb_channel(rgb_tensor, 1)
blue_only = isolate_rgb_channel(rgb_tensor, 2)

plt.figure(figsize=(14, 4))

plt.subplot(1, 4, 1)
plt.imshow(rgb_tensor.permute(1, 2, 0))
plt.title("Original RGB image")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(red_only.permute(1, 2, 0))
plt.title("Red channel contribution")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(green_only.permute(1, 2, 0))
plt.title("Green channel contribution")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(blue_only.permute(1, 2, 0))
plt.title("Blue channel contribution")
plt.axis("off")

plt.tight_layout()
plt.show()

## 1.5 Apply a simple transformation

Computer-vision models should work even when an object shifts slightly, rotates a bit, or appears in a different pose.

**Data augmentation** creates modified training examples, helping the model learn more general visual patterns.


In [ ]:
pil_rgb_image = TF.to_pil_image(rgb_tensor)

augmentation_demo = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(15),
])

augmented_rgb = augmentation_demo(pil_rgb_image)

plt.figure(figsize=(6, 6))
plt.imshow(augmented_rgb)
plt.title("Augmented RGB example")
plt.axis("off")
plt.show()

# Part 2 — CIFAR-10 image classification

CIFAR-10 contains 10 classes:

- airplane
- automobile
- bird
- cat
- deer
- dog
- frog
- horse
- ship
- truck

The goal is to give the model one image and ask it to predict **one class**.

## 2.1 Training and test transformations

### Why normalize?

Neural networks usually train more reliably when input values are on a consistent scale.

We normalize each RGB channel using the commonly used CIFAR-10 mean and standard deviation.

### Why use augmentation only for training?

Augmentation gives the model varied examples during learning.  
Validation and test images should remain unchanged so evaluation is fair and repeatable.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

## 2.2 Create the datasets

A PyTorch **Dataset** knows:

- how many examples exist
- how to retrieve one image and its label
- which transformations to apply

We use the official CIFAR-10 training split for learning and the official test split for final evaluation.

In [ ]:
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

class_names = train_dataset.classes

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", class_names)

## 2.3 Create a validation split

The three common data splits are:

- **Training set:** used to update the model.
- **Validation set:** used to check progress during training.
- **Test set:** used only for the final evaluation.

We create a validation subset from the original training set.

In [ ]:
validation_size = 5_000
training_size = len(train_dataset) - validation_size

generator = torch.Generator().manual_seed(SEED)

train_subset, validation_indices = torch.utils.data.random_split(
    range(len(train_dataset)),
    [training_size, validation_size],
    generator=generator,
)

train_indices = list(train_subset)
validation_indices = list(validation_indices)

validation_dataset_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=test_transform,
)

train_dataset_split = Subset(train_dataset, train_indices)
validation_dataset = Subset(validation_dataset_full, validation_indices)

print("Training subset:", len(train_dataset_split))
print("Validation subset:", len(validation_dataset))
print("Test set:", len(test_dataset))

## 2.4 Optional fast mode

Training a full model takes time.

- `FAST_MODE = True`: quicker demonstration
- `FAST_MODE = False`: use all available training data

The complete dataset should usually provide better results.

In [ ]:
FAST_MODE = False

if FAST_MODE:
    fast_train_size = 12_000
    fast_validation_size = 2_000

    train_dataset_split = Subset(
        train_dataset_split,
        range(min(fast_train_size, len(train_dataset_split))),
    )
    validation_dataset = Subset(
        validation_dataset,
        range(min(fast_validation_size, len(validation_dataset))),
    )

print("Images used for training:", len(train_dataset_split))
print("Images used for validation:", len(validation_dataset))

## 2.5 Create DataLoaders

A **DataLoader** takes examples from a Dataset and prepares them for the model.

It handles:

- grouping images into **batches**
- shuffling the training data
- loading data efficiently
- returning one batch at a time

A batch lets the GPU process several images together.

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset_split,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First labels:", labels[:10])

## 2.6 Visualize a batch from the DataLoader

Because the images have been normalized, we reverse the normalization before displaying them.

In [ ]:
mean_tensor = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std_tensor = torch.tensor(CIFAR10_STD).view(3, 1, 1)

def denormalize(image_tensor):
    image_tensor = image_tensor.cpu() * std_tensor + mean_tensor
    return image_tensor.clamp(0, 1)

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 7))

for index in range(12):
    plt.subplot(3, 4, index + 1)
    display_image = denormalize(images[index]).permute(1, 2, 0)
    plt.imshow(display_image)
    plt.title(class_names[labels[index].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()

# Part 3 — Build a ResNet classifier

A very deep network can become difficult to optimize. ResNet introduces **residual connections** that let information skip over layers.

Conceptually:

```text
input ────────────────┐
   │                  │
   └→ learned layers ─┴→ output
```

We use ResNet-18, but make two small changes for CIFAR-10:

1. Replace the large first convolution with a `3 × 3` convolution.
2. Remove the early max-pooling layer.

These changes preserve more detail in tiny `32 × 32` images.

In [ ]:
model = models.resnet18(weights=None)

model.conv1 = nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False,
)

model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, len(class_names))

model = model.to(device)

print(model)

## 3.1 Check the model output

For a batch of images, the model returns one score for each class.

With batch size 128 and 10 classes, the output shape is:

```text
128 × 10
```

These raw scores are called **logits**.

In [ ]:
sample_images = images[:4].to(device)

with torch.no_grad():
    sample_logits = model(sample_images)

print("Input shape:", sample_images.shape)
print("Output shape:", sample_logits.shape)

# Part 4 — Loss function and optimization

## Loss function

The **loss function** measures how wrong the model is.

For multiclass classification, we use `CrossEntropyLoss`.

It compares:

- the model's 10 class scores
- the correct class index

Lower loss generally means the predictions are improving.

> `CrossEntropyLoss` expects raw logits, so we do **not** add a Softmax layer inside the model.

## Optimizer

The **optimizer** changes the model's weights to reduce the loss.

We use AdamW. Its learning rate controls the size of each update:

- too large: training may become unstable
- too small: learning may be very slow

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10,
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

# Part 5 — The training loop

For every batch:

1. Move images and labels to the GPU or CPU.
2. Clear gradients from the previous batch.
3. Run the model to obtain predictions.
4. Calculate the loss.
5. Use backpropagation to calculate gradients.
6. Let the optimizer update the model weights.
7. Record loss and accuracy.

The validation loop is similar, but it does **not** update the model.

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_examples = 0

    for images, labels in data_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # 1. Remove gradients left from the previous batch.
        optimizer.zero_grad(set_to_none=True)

        # 2. Forward pass: compute class scores.
        logits = model(images)

        # 3. Measure prediction error.
        loss = criterion(logits, labels)

        # 4. Backward pass: calculate gradients.
        loss.backward()

        # 5. Update the weights.
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        predictions = logits.argmax(dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_examples += batch_size

    epoch_loss = running_loss / total_examples
    epoch_accuracy = correct_predictions / total_examples

    return epoch_loss, epoch_accuracy


@torch.no_grad()
def evaluate(model, data_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_examples = 0

    for images, labels in data_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size

        predictions = logits.argmax(dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_examples += batch_size

    average_loss = running_loss / total_examples
    accuracy = correct_predictions / total_examples

    return average_loss, accuracy

## 5.1 Train the model

An **epoch** is one complete pass through the training data.



In [ ]:
EPOCHS = 15

history = {
    "train_loss": [],
    "train_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
}

best_validation_accuracy = 0.0
best_model_path = Path("best_resnet18_cifar10.pth")

training_start_time = time.time()

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )

    validation_loss, validation_accuracy = evaluate(
        model,
        validation_loader,
        criterion,
        device,
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["validation_loss"].append(validation_loss)
    history["validation_accuracy"].append(validation_accuracy)

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save(model.state_dict(), best_model_path)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy * 100:6.2f}% | "
        f"Val loss: {validation_loss:.4f} | "
        f"Val acc: {validation_accuracy * 100:6.2f}%"
    )

elapsed_minutes = (time.time() - training_start_time) / 60

print(f"\nTraining time: {elapsed_minutes:.2f} minutes")
print(f"Best validation accuracy: {best_validation_accuracy * 100:.2f}%")
print("Saved model:", best_model_path)

## 5.2 Plot the learning curves

Learning curves help us answer:

- Is the model learning?
- Is validation performance improving?
- Is there a large gap between training and validation accuracy?
- Could the model be overfitting?

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs, history["train_loss"], marker="o", label="Training loss")
plt.plot(epochs, history["validation_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(
    epochs,
    np.array(history["train_accuracy"]) * 100,
    marker="o",
    label="Training accuracy",
)
plt.plot(
    epochs,
    np.array(history["validation_accuracy"]) * 100,
    marker="o",
    label="Validation accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Part 6 — Final test evaluation

The test set is used after training decisions are complete.

We first load the model version that achieved the best validation accuracy.

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))

test_loss, test_accuracy = evaluate(
    model,
    test_loader,
    criterion,
    device,
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

## 6.1 Visualize predictions

Green titles represent correct predictions.  
Red titles represent incorrect predictions.

In [ ]:
model.eval()

test_images, test_labels = next(iter(test_loader))

with torch.no_grad():
    test_logits = model(test_images.to(device))
    test_predictions = test_logits.argmax(dim=1).cpu()

plt.figure(figsize=(12, 9))

for index in range(16):
    plt.subplot(4, 4, index + 1)

    display_image = denormalize(test_images[index]).permute(1, 2, 0)
    true_name = class_names[test_labels[index].item()]
    predicted_name = class_names[test_predictions[index].item()]

    title_color = "green" if true_name == predicted_name else "red"

    plt.imshow(display_image)
    plt.title(
        f"Pred: {predicted_name}\nTrue: {true_name}",
        color=title_color,
        fontsize=9,
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 6.2 Inspect class probabilities for one image

The model produces a score for each class. Softmax converts the scores into probabilities that add up to 1.

In [ ]:
example_index = 0
example_image = test_images[example_index].unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(example_image)
    probabilities = torch.softmax(logits, dim=1)[0].cpu()

top_probabilities, top_indices = torch.topk(probabilities, k=5)

plt.figure(figsize=(3, 3))
plt.imshow(denormalize(test_images[example_index]).permute(1, 2, 0))
plt.title(f"True class: {class_names[test_labels[example_index].item()]}")
plt.axis("off")
plt.show()

for rank, (probability, class_index) in enumerate(
    zip(top_probabilities, top_indices),
    start=1,
):
    print(
        f"{rank}. {class_names[class_index.item()]:10s} "
        f"{probability.item() * 100:6.2f}%"
    )

## 6.3 Discussion: Why can the model fail?

Look for examples involving:

- low resolution
- unusual viewpoints
- similar-looking classes
- background clutter
- partial visibility
- limited training time
- insufficient model capacity
- mismatch between training and test examples

A model's failure is evidence we can study—not merely a bad score.

# Part 7 — Object detection with a pretrained YOLO model

Image classification answers:

> **What is the main object in this image?**

Object detection answers:

> **What objects are present, and where is each object?**

A detector returns:

- class labels
- confidence scores
- bounding boxes

In this section, we do not train YOLO. We use a pretrained model to get a quick overview of object detection.

## 7.1 Install Ultralytics

Ultralytics YOLO models are implemented with PyTorch.

The package and pretrained weights require internet access the first time they are downloaded.

In [ ]:
!pip install -q -U ultralytics

## 7.2 Load a pretrained YOLO detector

The `n` model is the small “nano” version, which is convenient for a classroom demonstration.

In [ ]:
from ultralytics import YOLO

detector = YOLO("yolo11n.pt")

print("YOLO model loaded.")

## 7.3 Try YOLO on the built-in COCO8 mini dataset

`coco8.yaml` is a tiny example dataset supported by Ultralytics. It contains a few images in the same label format used by the larger COCO object-detection dataset.

This cell automatically downloads the mini dataset if needed and runs validation with the pretrained detector. The purpose is to see the object-detection workflow—not to obtain a meaningful benchmark from only a few images.

In [ ]:
coco8_metrics = detector.val(
    data="coco8.yaml",
    imgsz=640,
    batch=1,
    verbose=False,
)

print("COCO8 validation complete.")
print("mAP50:", float(coco8_metrics.box.map50))
print("mAP50-95:", float(coco8_metrics.box.map))

## 7.4 Run detection on a sample street image



In [ ]:
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

sample_source = "https://ultralytics.com/images/bus.jpg"

# Download and display the original image
response = requests.get(sample_source, timeout=30)
response.raise_for_status()

original_image = Image.open(BytesIO(response.content)).convert("RGB")

plt.figure(figsize=(7, 5))
plt.imshow(original_image)
plt.title("Original image")
plt.axis("off")
plt.show()

# Run pretrained YOLO detection
detection_results = detector.predict(
    source=sample_source,
    conf=0.25,
    save=False,
    verbose=False,
)

result = detection_results[0]
annotated_image = result.plot()

# Ultralytics returns the plotted image in BGR format
annotated_image_rgb = annotated_image[:, :, ::-1]

plt.figure(figsize=(7, 5))
plt.imshow(annotated_image_rgb)
plt.title("Pretrained YOLO detections")
plt.axis("off")
plt.show()

## 7.5 Inspect the detected classes and confidence scores

In [ ]:
boxes = result.boxes

if boxes is None or len(boxes) == 0:
    print("No objects detected.")
else:
    for class_id, confidence in zip(
        boxes.cls.cpu().to(torch.int64),
        boxes.conf.cpu(),
    ):
        class_name = detector.names[class_id.item()]
        print(f"{class_name:12s} confidence: {confidence.item():.3f}")

## 7.6 Count objects by class

In [ ]:
from collections import Counter

if boxes is not None and len(boxes) > 0:
    detected_names = [
        detector.names[class_id.item()]
        for class_id in boxes.cls.cpu().to(torch.int64)
    ]

    counts = Counter(detected_names)

    print("Detected object counts:")
    for class_name, count in counts.items():
        print(f"- {class_name}: {count}")

## 7.7 Optional: try another image

Replace `source` with:

- a local file path
- an image URL
- a video path
- a webcam source supported by the environment

In Google Colab, uploading an image is often easiest.

In [ ]:
# Optional Colab upload:
#
# from google.colab import files
# uploaded = files.upload()
# uploaded_filename = next(iter(uploaded))
#
# custom_results = detector.predict(
#     source=uploaded_filename,
#     conf=0.25,
#     verbose=False,
# )
#
# custom_annotated = custom_results[0].plot()[:, :, ::-1]
#
# plt.figure(figsize=(10, 7))
# plt.imshow(custom_annotated)
# plt.axis("off")
# plt.show()

# Final reflection

Discuss these questions with your group:

1. What does an image tensor contain?
2. What is the difference between a Dataset and a DataLoader?
3. Why do we separate training, validation, and test data?
4. What does the loss function measure?
5. What does the optimizer change?
6. Why do we call `optimizer.zero_grad()`?
7. What is the difference between classification and object detection?
8. Which failure cases would you test before deploying a traffic detector?

